# Figure 0 — title figure

A compact two-panel version of the posterior collapse for the title slide: everything you believe before any experiment, and what six experiments leave standing.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "_shared"))
sys.path.insert(0, str(pathlib.Path.cwd() / "_shared"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import style, gp as gpmod, landscape as land, doe as doemod
style.use_deck_style()
OUT = "../lecture_12_figures/generated"

In [ ]:

xs = np.linspace(0, 10, 500)
# acquisition order, not left-to-right: a real campaign jumps around the
# domain (space-filling, then homing in) rather than sweeping monotonically
OX = np.array([4.30, 8.90, 1.15, 6.10, 2.90, 7.40])
OY = land.f1d(OX)
LS, SF, SN = 0.85, 1.0, 0.03   # Matern-5/2 length-scale, signal sd, noise sd

fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.5), sharey=True,
                         gridspec_kw=dict(wspace=0.08))
for ax, k, lab in zip(axes, [0, 6],
                      ["before any experiment", "after six experiments"]):
    g = gpmod.GP(gpmod.matern52, ls=LS, sf=SF, sn=SN)
    if k:
        g.fit(OX[:k, None], OY[:k])
    mu, sd = g.predict(xs[:, None])
    for s in g.sample(xs[:, None], n=90, seed=7 + k):
        ax.plot(xs, s, color=style.TEAL, lw=0.35, alpha=0.20)
    ax.fill_between(xs, mu - 2 * sd, mu + 2 * sd, color=style.TEAL,
                    alpha=0.13, lw=0)
    ax.plot(xs, mu, color=style.INK, lw=1.8, alpha=0.55 if k == 0 else 1.0)
    if k:
        ax.plot(OX[:k], OY[:k], "o", ms=7, color=style.RED, mec="white", mew=1.1)
        i = int(np.argmax(OY[:k]))
        ax.annotate("best so far", (OX[i], OY[i]), textcoords="offset points",
                    xytext=(6, 12), fontsize=9, color=style.RED)
    ax.set_ylim(-2.6, 2.6)
    ax.set_xlabel("reaction parameter  x")
    ax.set_title(lab, loc="left", fontsize=11)
    ax.text(0.98, 0.04, fr"$\ell$={LS}   $\sigma_f$={SF}   $\sigma_n$={SN}",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=8.5, color=style.GRAY)
axes[0].set_ylabel("objective")
pdf_path, png_path = style.save(fig, "fig_00_title_gp_collapse", OUT)

from IPython.display import Image, display
display(Image(filename=png_path))

## Animation — watching the acquisition happen

The two static panels only show the endpoints. This animation steps through the six experiments one at a time: each new observation (gold) is folded into the fit, the credible band tightens locally around it, and the sample paths that disagree with it are deleted. Same kernel and hyperparameters as above.


In [ ]:

from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display

fig_a, ax_a = plt.subplots(figsize=(5.4, 3.7))

def _draw(k):
    ax_a.clear()
    g = gpmod.GP(gpmod.matern52, ls=LS, sf=SF, sn=SN)
    if k:
        g.fit(OX[:k, None], OY[:k])
    mu, sd = g.predict(xs[:, None])
    for s in g.sample(xs[:, None], n=60, seed=7 + k):
        ax_a.plot(xs, s, color=style.TEAL, lw=0.35, alpha=0.18)
    ax_a.fill_between(xs, mu - 2 * sd, mu + 2 * sd, color=style.TEAL,
                      alpha=0.13, lw=0)
    ax_a.plot(xs, mu, color=style.INK, lw=1.8, alpha=0.55 if k == 0 else 1.0)
    if k:
        ax_a.plot(OX[:k - 1], OY[:k - 1], "o", ms=7, color=style.RED,
                  mec="white", mew=1.1, zorder=6)
        ax_a.plot(OX[k - 1:k], OY[k - 1:k], "o", ms=10, color=style.GOLD,
                  mec="white", mew=1.3, zorder=7)
    ax_a.set_xlim(0, 10)
    ax_a.set_ylim(-2.6, 2.6)
    ax_a.set_xlabel("reaction parameter  x")
    ax_a.set_ylabel("objective")
    title = "before any experiment" if k == 0 else f"after experiment {k}"
    ax_a.set_title(title, loc="left", fontsize=11)
    ax_a.text(0.98, 0.04, fr"$\ell$={LS}   $\sigma_f$={SF}   $\sigma_n$={SN}",
              transform=ax_a.transAxes, ha="right", va="bottom",
              fontsize=8.5, color=style.GRAY)

anim = FuncAnimation(fig_a, _draw, frames=range(7), interval=900)
gif_path = f"{OUT}/fig_00_title_gp_acquisition.gif"
anim.save(gif_path, writer=PillowWriter(fps=1))
plt.close(fig_a)
print("wrote", gif_path)
display(Image(filename=gif_path))